In [18]:
!pip install tensorflow -q
!pip install matplotlib -q
!pip install pandas -q
!pip install torch -q


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [19]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, BatchNormalization, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [20]:
train_dir = tf.keras.utils.image_dataset_from_directory(
    "/Users/ashishjadia/LABMENTIX/Brain_Tumor/train",
    image_size=(224, 224),
    batch_size=32,
    label_mode='categorical'
)

val_dir = tf.keras.utils.image_dataset_from_directory(
    "/Users/ashishjadia/LABMENTIX/Brain_Tumor/valid",
    image_size=(224, 224),
    batch_size=32,
    label_mode='categorical'
)

test_dir = tf.keras.utils.image_dataset_from_directory(
    "/Users/ashishjadia/LABMENTIX/Brain_Tumor/test",
    image_size=(224, 224),
    batch_size=32,
    label_mode='categorical'
)

num_classes = len(train_dir.class_names)

Found 1695 files belonging to 4 classes.
Found 502 files belonging to 4 classes.
Found 246 files belonging to 4 classes.


In [21]:
train_csv = pd.read_csv("/Users/ashishjadia/LABMENTIX/Brain_Tumor/train/_classes.csv")
val_csv = pd.read_csv("/Users/ashishjadia/LABMENTIX/Brain_Tumor/valid/_classes.csv")
test_csv = pd.read_csv("/Users/ashishjadia/LABMENTIX/Brain_Tumor/test/_classes.csv")

In [22]:
train_csv, val_csv, test_csv

(                                               filename   Glioma   Meningioma  \
 0     Tr-pi_0164_jpg.rf.000776527ec0acdc89e31e15a352...        0            0   
 1     Tr-no_0426_jpg.rf.0026b06f369c5d51aca4c4c9beba...        0            0   
 2     Tr-gl_0496_jpg.rf.010620fbbbaa509aa81d7ce5bdf7...        1            0   
 3     Tr-gl_0554_jpg.rf.010a72c1c25cc9ce83c77fbb23db...        1            0   
 4     Tr-me_0185_jpg.rf.0094b0b539582e2f95ae7b6ada4d...        0            1   
 ...                                                 ...      ...          ...   
 1690  Tr-pi_0266_jpg.rf.be96620d2e87b7d1278a7cdebc31...        0            0   
 1691  Tr-no_0474_jpg.rf.bf5869c9603ce71575b58df92ca1...        0            0   
 1692  Tr-me_0356_jpg.rf.be4ba7e5211c086bdfb21a53ee6b...        0            1   
 1693  Tr-gl_0169_jpg.rf.bf25b05a8345a96e51560f67eed7...        1            0   
 1694  Tr-pi_0167_jpg.rf.bf3a10b82161ca3042c8e481b9c3...        0            0   
 
        No Tum

In [23]:
AUTOTUNE = tf.data.AUTOTUNE

train_dir = train_dir.cache().prefetch(buffer_size=AUTOTUNE).shuffle(buffer_size=1000)
val_dir = val_dir.cache().prefetch(buffer_size=AUTOTUNE)
test_dir = test_dir.cache().prefetch(buffer_size=AUTOTUNE)

In [24]:
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom, RandomBrightness, Lambda

data_augmentation = tf.keras.Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.1),
    RandomZoom(0.1),
    RandomBrightness(0.1)
])

In [25]:
preprocess_input = Lambda(preprocess_input)

In [26]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

base_model.trainable = False

In [ ]:
custom_model = tf.keras.Sequential([
    data_augmentation,
    preprocess_input,
    base_model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dropout(0.5),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

In [36]:
custom_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [37]:
custom_model.fit(
    train_dir,
    validation_data=val_dir,
    epochs=50,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
        ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True)
    ]
)

Epoch 1/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 45s 770ms/step - accuracy: 0.8100 - loss: 0.6809 - val_accuracy: 0.2689 - val_loss: 289.4054
Epoch 2/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 43s 808ms/step - accuracy: 0.9133 - loss: 0.2677 - val_accuracy: 0.7072 - val_loss: 6.0043
Epoch 3/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 45s 847ms/step - accuracy: 0.9363 - loss: 0.1874 - val_accuracy: 0.8486 - val_loss: 1.2589
Epoch 4/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 46s 872ms/step - accuracy: 0.9569 - loss: 0.1243 - val_accuracy: 0.8466 - val_loss: 2.9717
Epoch 5/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 49s 928ms/step - accuracy: 0.9540 - loss: 0.1241 - val_accuracy: 0.9382 - val_loss: 0.2586
Epoch 6/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 51s 961ms/step - accuracy: 0.9628 - loss: 0.1028 - val_accuracy: 0.9323 - val_loss: 0.3760
Epoch 7/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 50s 951ms/step - accuracy: 0.9693 - loss: 0.0815 - val_accuracy: 0.9203 - val_loss: 0.2835
Epoch 8/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 50s 940ms/step - accuracy: 0.9611 - loss: 0.1363 - val_ac

In [38]:
custom_model.evaluate(test_dir)

8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 257ms/step - accuracy: 0.9390 - loss: 0.2196


[0.21955500543117523, 0.9390243887901306]

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:100]:
    layer.trainable = False

custom_model1 = tf.keras.Sequential([
    data_augmentation,
    preprocess_input,
    base_model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dropout(0.5),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

In [41]:
custom_model1.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

In [42]:
custom_model1.fit(
    train_dir,
    validation_data=val_dir,
    epochs=50,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
        ModelCheckpoint('best_model_finetuned.keras', monitor='val_loss', save_best_only=True)
    ])

Epoch 1/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 45s 765ms/step - accuracy: 0.4956 - loss: 1.3647 - val_accuracy: 0.8446 - val_loss: 0.4691
Epoch 2/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 43s 811ms/step - accuracy: 0.8088 - loss: 0.5209 - val_accuracy: 0.9283 - val_loss: 0.2586
Epoch 3/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 45s 848ms/step - accuracy: 0.8791 - loss: 0.3209 - val_accuracy: 0.9422 - val_loss: 0.1903
Epoch 4/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 46s 872ms/step - accuracy: 0.9322 - loss: 0.1972 - val_accuracy: 0.9402 - val_loss: 0.1708
Epoch 5/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 49s 924ms/step - accuracy: 0.9363 - loss: 0.1850 - val_accuracy: 0.9402 - val_loss: 0.1693
Epoch 6/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 50s 948ms/step - accuracy: 0.9445 - loss: 0.1669 - val_accuracy: 0.9422 - val_loss: 0.1776
Epoch 7/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 50s 940ms/step - accuracy: 0.9634 - loss: 0.1130 - val_accuracy: 0.9442 - val_loss: 0.1839
Epoch 8/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 49s 932ms/step - accuracy: 0.9676 - loss: 0.1069 - val_accu

In [44]:
custom_model1.evaluate(test_dir)

8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 268ms/step - accuracy: 0.9390 - loss: 0.1857


[0.1857338547706604, 0.9390243887901306]